📁 결과물
출력:

N:\개인\이수빈\3.13_Mini_Project\evaluation\yolov8n_test\
    ㄴtest_results.csv - 성능 지표    
    ㄴtest_results.json - 상세 결과    
    ㄴfalse_negatives/ - 미탐 이미지 (NOT DETECTED 표시)    
    ㄴfalse_negatives/originals/ - 미탐 원본    
    ㄴfalse_positives/ - 오탐 이미지 (bbox 표시)    
    ㄴfalse_positives/originals/ - 오탐 원본    

셀 1: 라이브러리 import

In [1]:
# ============================================
# Test Set 평가 + 시각화
# ============================================

# YOLO 모델
from ultralytics import YOLO

# 경로 처리
from pathlib import Path

# 이미지 처리 및 bbox 그리기
import cv2

# 파일 복사
import shutil

# 데이터 저장
import pandas as pd

# JSON 저장
import json

# 날짜/시간
from datetime import datetime

print("✅ 라이브러리 import 완료")

✅ 라이브러리 import 완료


셀 2: 경로 설정

In [2]:
# ============================================
# 경로 설정
# ============================================

# 프로젝트 루트 경로
PROJECT_ROOT = Path(r'N:\개인\이수빈\3.13_Mini_Project')

# 모델 경로
MODEL_PATH = PROJECT_ROOT / 'results' / 'yolov8n_70_15' / 'weights' / 'best.pt'

# Test set 경로
TEST_DIR = PROJECT_ROOT / 'DATASET' / 'test_set_894'

# 결과 저장 경로
EVAL_DIR = PROJECT_ROOT / 'evaluation' / 'yolov8n_test'

# 하위 폴더 생성
(EVAL_DIR / 'false_negatives').mkdir(parents=True, exist_ok=True)  # 미탐 (bbox 표시)
(EVAL_DIR / 'false_negatives' / 'originals').mkdir(parents=True, exist_ok=True)  # 미탐 원본
(EVAL_DIR / 'false_positives').mkdir(parents=True, exist_ok=True)  # 오탐 (bbox 표시)
(EVAL_DIR / 'false_positives' / 'originals').mkdir(parents=True, exist_ok=True)  # 오탐 원본

print("=" * 60)
print("📊 YOLOv8n Test Set 평가")
print("=" * 60)
print(f"\n모델: {MODEL_PATH}")
print(f"Test: {TEST_DIR}")
print(f"결과: {EVAL_DIR}")

📊 YOLOv8n Test Set 평가

모델: N:\개인\이수빈\3.13_Mini_Project\results\yolov8n_70_15\weights\best.pt
Test: N:\개인\이수빈\3.13_Mini_Project\DATASET\test_set_894
결과: N:\개인\이수빈\3.13_Mini_Project\evaluation\yolov8n_test


 셀 3: 모델 로드 및 Test 데이터 확인

In [3]:
# ============================================
# 모델 로드 및 Test 데이터 확인
# ============================================

print("\n[모델 로드]")

# Best 모델 로드
model = YOLO(str(MODEL_PATH))
print(f"✅ 모델 로드 완료")

# Test 이미지 리스트 (화재)
test_fire = [f for f in (TEST_DIR / 'fire').iterdir() 
             if f.suffix.lower() in ['.jpg', '.jpeg', '.png', '.webp']]

# Test 이미지 리스트 (정상)
test_normal = [f for f in (TEST_DIR / 'normal').iterdir() 
               if f.suffix.lower() in ['.jpg', '.jpeg', '.png', '.webp']]

# 개수 출력
print(f"\n[Test 데이터]")
print(f"  화재:  {len(test_fire)}장")
print(f"  정상:  {len(test_normal)}장")
print(f"  총:    {len(test_fire) + len(test_normal)}장")

print("=" * 60)


[모델 로드]
✅ 모델 로드 완료

[Test 데이터]
  화재:  447장
  정상:  447장
  총:    894장


셀 4: Test 평가 실행

In [4]:
# ============================================
# Test 평가 실행
# ============================================

print("\n[평가 시작]")
print(f"시작: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# Confidence threshold 설정
CONF_THRESHOLD = 0.20

# True Positive 카운터 (화재를 화재로 맞춤)
true_positive = 0

# False Negative 카운터 (화재인데 못 찾음 = 미탐) ⚠️
false_negative = 0

# False Negative 리스트 (이미지 경로 저장)
fn_list = []

# ============================================
# 1. 화재 이미지 평가
# ============================================

print(f"\n[화재 이미지 평가 중...]")

# 각 화재 이미지에 대해 반복
for idx, img_path in enumerate(test_fire, 1):
    # 예측 실행
    results = model.predict(
        str(img_path),         # 이미지 경로
        conf=CONF_THRESHOLD,   # Confidence threshold
        imgsz=640,             # 이미지 크기
        save=False,            # 자동 저장 안 함 (우리가 직접 저장)
        verbose=False          # 출력 최소화
    )
    
    # bbox 개수 확인
    num_boxes = len(results[0].boxes)
    
    # 탐지 여부 확인
    if num_boxes > 0:
        # 탐지 성공 (True Positive)
        true_positive += 1
    else:
        # 탐지 실패 (False Negative = 미탐!) ⚠️
        false_negative += 1
        
        # 미탐 이미지 경로 저장
        fn_list.append(img_path)
        
        # 원본 이미지 로드
        img = cv2.imread(str(img_path))
        
        # "NOT DETECTED" 텍스트 표시
        cv2.putText(
            img,                          # 이미지
            'NOT DETECTED',               # 텍스트
            (50, 50),                     # 위치 (x, y)
            cv2.FONT_HERSHEY_SIMPLEX,     # 폰트
            1.5,                          # 크기
            (0, 0, 255),                  # 빨강색 (BGR)
            3                             # 두께
        )
        
        # 텍스트 표시된 이미지 저장
        save_path = EVAL_DIR / 'false_negatives' / f"FN_{img_path.name}"
        cv2.imwrite(str(save_path), img)
        
        # 원본도 복사
        orig_path = EVAL_DIR / 'false_negatives' / 'originals' / img_path.name
        shutil.copy2(img_path, orig_path)
    
    # 진행 상황 (50장마다)
    if idx % 50 == 0:
        print(f"  {idx}/{len(test_fire)}")

print(f"✅ 화재 평가 완료")
print(f"   탐지: {true_positive}장")
print(f"   미탐: {false_negative}장")

# ============================================
# 2. 정상 이미지 평가 (오탐 확인)
# ============================================

print(f"\n[정상 이미지 평가 중...]")

# True Negative 카운터 (정상을 정상으로 맞춤)
true_negative = 0

# False Positive 카운터 (정상인데 화재로 오인 = 오탐)
false_positive = 0

# False Positive 리스트
fp_list = []

# 각 정상 이미지에 대해 반복
for idx, img_path in enumerate(test_normal, 1):
    # 예측 실행
    results = model.predict(
        str(img_path),         # 이미지 경로
        conf=CONF_THRESHOLD,   # Confidence threshold
        imgsz=640,             # 이미지 크기
        save=False,            # 자동 저장 안 함
        verbose=False          # 출력 최소화
    )
    
    # bbox 개수 확인
    num_boxes = len(results[0].boxes)
    
    # 탐지 여부 확인
    if num_boxes > 0:
        # 탐지됨 (False Positive = 오탐!)
        false_positive += 1
        
        # 오탐 이미지 경로 저장
        fp_list.append(img_path)
        
        # bbox 그려진 이미지 생성
        result_img = results[0].plot()  # bbox, label, conf 자동 표시
        
        # bbox 그려진 이미지 저장
        save_path = EVAL_DIR / 'false_positives' / f"FP_{img_path.name}"
        cv2.imwrite(str(save_path), result_img)
        
        # 원본도 복사
        orig_path = EVAL_DIR / 'false_positives' / 'originals' / img_path.name
        shutil.copy2(img_path, orig_path)
    else:
        # 탐지 안 됨 (True Negative = 정상 맞춤)
        true_negative += 1
    
    # 진행 상황
    if idx % 50 == 0:
        print(f"  {idx}/{len(test_normal)}")

print(f"✅ 정상 평가 완료")
print(f"   정상: {true_negative}장")
print(f"   오탐: {false_positive}장")

print(f"\n평가 완료: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 60)


[평가 시작]
시작: 2026-03-02 02:55:19

[화재 이미지 평가 중...]
  50/447
  100/447
  150/447
  200/447
  250/447
  300/447
  350/447
  400/447
✅ 화재 평가 완료
   탐지: 409장
   미탐: 38장

[정상 이미지 평가 중...]
  50/447
  100/447
  150/447
  200/447
  250/447
  300/447
  350/447
  400/447
✅ 정상 평가 완료
   정상: 432장
   오탐: 15장

평가 완료: 2026-03-02 02:56:19


셀 5: 성능 계산 및 출력

In [5]:
# ============================================
# 성능 지표 계산
# ============================================

print("\n[성능 지표 계산]")

# Recall (재현율) = TP / (TP + FN)
# 실제 화재 중 찾은 비율
recall = true_positive / (true_positive + false_negative) if (true_positive + false_negative) > 0 else 0

# Precision (정밀도) = TP / (TP + FP)
# 모델이 화재라고 예측한 것 중 맞은 비율
precision = true_positive / (true_positive + false_positive) if (true_positive + false_positive) > 0 else 0

# F1 Score = 2 * (Precision * Recall) / (Precision + Recall)
# Precision과 Recall의 조화평균
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

# ============================================
# 결과 출력
# ============================================

print(f"\n{'='*60}")
print("📊 YOLOv8n Test Set 최종 성능")
print(f"{'='*60}")

# Confusion Matrix
print(f"\n혼동 행렬:")
print(f"                실제 화재    실제 정상")
print(f"  예측 화재    {true_positive:>6}       {false_positive:>6}  ")
print(f"  예측 정상    {false_negative:>6}       {true_negative:>6}  ")

# 주요 지표
print(f"\n주요 지표:")
print(f"  Recall:    {recall:.3f}  (목표: 0.920)")
print(f"  Precision: {precision:.3f}  (목표: 0.880)")
print(f"  F1 Score:  {f1_score:.3f}")

# 상세 분석
print(f"\n상세 분석:")
print(f"  총 화재:   {len(test_fire)}장")
print(f"    탐지:    {true_positive}장 ({true_positive/len(test_fire)*100:.1f}%)")
print(f"    미탐:    {false_negative}장 ({false_negative/len(test_fire)*100:.1f}%) ⚠️")

print(f"\n  총 정상:   {len(test_normal)}장")
print(f"    정상:    {true_negative}장 ({true_negative/len(test_normal)*100:.1f}%)")
print(f"    오탐:    {false_positive}장 ({false_positive/len(test_normal)*100:.1f}%)")

# 목표 달성 확인
print(f"\n{'='*60}")
print("🎯 목표 달성 여부")
print(f"{'='*60}")

if recall >= 0.92:
    print(f"✅ Recall 달성! {recall:.3f} (목표 0.920, +{(recall-0.92)*100:.1f}%p)")
else:
    print(f"⚠️ Recall 부족: {recall:.3f} (목표 0.920, {(0.92-recall)*100:.1f}%p 부족)")

if precision >= 0.88:
    print(f"✅ Precision 달성! {precision:.3f} (목표 0.880, +{(precision-0.88)*100:.1f}%p)")
else:
    print(f"⚠️ Precision 부족: {precision:.3f} (목표 0.880, {(0.88-precision)*100:.1f}%p 부족)")

print(f"\n{'='*60}")


[성능 지표 계산]

📊 YOLOv8n Test Set 최종 성능

혼동 행렬:
                실제 화재    실제 정상
  예측 화재       409           15  
  예측 정상        38          432  

주요 지표:
  Recall:    0.915  (목표: 0.920)
  Precision: 0.965  (목표: 0.880)
  F1 Score:  0.939

상세 분석:
  총 화재:   447장
    탐지:    409장 (91.5%)
    미탐:    38장 (8.5%) ⚠️

  총 정상:   447장
    정상:    432장 (96.6%)
    오탐:    15장 (3.4%)

🎯 목표 달성 여부
⚠️ Recall 부족: 0.915 (목표 0.920, 0.5%p 부족)
✅ Precision 달성! 0.965 (목표 0.880, +8.5%p)



셀 6: 결과 저장

In [6]:
# ============================================
# 결과 저장 (CSV + JSON)
# ============================================

print("\n[결과 저장]")

# CSV 저장용 데이터
results_data = {
    'Model': 'YOLOv8n',
    'Test_Fire': len(test_fire),
    'Test_Normal': len(test_normal),
    'True_Positive': true_positive,
    'False_Negative': false_negative,
    'True_Negative': true_negative,
    'False_Positive': false_positive,
    'Recall': recall,
    'Precision': precision,
    'F1_Score': f1_score,
    'Conf_Threshold': CONF_THRESHOLD,
    'Evaluated_At': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
}

# DataFrame 생성 및 CSV 저장
df = pd.DataFrame([results_data])
csv_path = EVAL_DIR / 'test_results.csv'
df.to_csv(csv_path, index=False, encoding='utf-8-sig')
print(f"✅ CSV 저장: {csv_path}")

# JSON 저장용 데이터 (상세 정보 포함)
json_data = {
    **results_data,
    'false_negatives': [str(p) for p in fn_list],  # 미탐 파일 목록
    'false_positives': [str(p) for p in fp_list]   # 오탐 파일 목록
}

# JSON 저장
json_path = EVAL_DIR / 'test_results.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(json_data, f, indent=2, ensure_ascii=False)
print(f"✅ JSON 저장: {json_path}")

# ============================================
# 저장 위치 안내
# ============================================

print(f"\n{'='*60}")
print("📁 결과 파일 위치")
print(f"{'='*60}")

print(f"\n결과:")
print(f"  {csv_path}")
print(f"  {json_path}")

print(f"\n미탐 ({false_negative}건):")
print(f"  bbox 표시: {EVAL_DIR / 'false_negatives'}")
print(f"  원본:     {EVAL_DIR / 'false_negatives' / 'originals'}")

print(f"\n오탐 ({false_positive}건):")
print(f"  bbox 표시: {EVAL_DIR / 'false_positives'}")
print(f"  원본:     {EVAL_DIR / 'false_positives' / 'originals'}")

print(f"\n{'='*60}")
print("✅ 평가 완료!")
print(f"{'='*60}")


[결과 저장]
✅ CSV 저장: N:\개인\이수빈\3.13_Mini_Project\evaluation\yolov8n_test\test_results.csv
✅ JSON 저장: N:\개인\이수빈\3.13_Mini_Project\evaluation\yolov8n_test\test_results.json

📁 결과 파일 위치

결과:
  N:\개인\이수빈\3.13_Mini_Project\evaluation\yolov8n_test\test_results.csv
  N:\개인\이수빈\3.13_Mini_Project\evaluation\yolov8n_test\test_results.json

미탐 (38건):
  bbox 표시: N:\개인\이수빈\3.13_Mini_Project\evaluation\yolov8n_test\false_negatives
  원본:     N:\개인\이수빈\3.13_Mini_Project\evaluation\yolov8n_test\false_negatives\originals

오탐 (15건):
  bbox 표시: N:\개인\이수빈\3.13_Mini_Project\evaluation\yolov8n_test\false_positives
  원본:     N:\개인\이수빈\3.13_Mini_Project\evaluation\yolov8n_test\false_positives\originals

✅ 평가 완료!


코드 결과를 보니 mAP지표가 없음.

 셀 7: mAP 계산 + 데이터 상세 분석 (날씨 포함)

In [9]:
# ============================================
# Validation mAP + Test 성능 + 데이터 분석
# ============================================

print("\n[성능 지표 확인]")
print("=" * 60)

# ============================================
# 1. Validation mAP (학습 결과에서 가져오기)
# ============================================

print("\nValidation mAP (학습 중 계산됨)...")

# results.csv 파일 읽기 (학습 중 저장된 결과)
results_csv = PROJECT_ROOT / 'results' / 'yolov8n_70_15' / 'results.csv'

# CSV 파일이 있는지 확인
if results_csv.exists():
    # pandas로 CSV 읽기
    import pandas as pd
    df = pd.read_csv(results_csv)
    
    # 마지막 에폭 결과 (Best 모델 기준, 또는 마지막 행)
    # CSV 열 이름 확인 필요 (공백 있을 수 있음)
    
    # metrics/mAP50(B) 또는 유사한 열 찾기
    # 열 이름 출력 (디버깅용)
    print(f"  CSV 열: {list(df.columns)}")
    
    # mAP50 찾기 (열 이름 패턴 매칭)
    map50_col = [col for col in df.columns if 'mAP50' in col or 'map50' in col]
    map50_95_col = [col for col in df.columns if 'mAP50-95' in col or 'map' in col and '95' in col]
    
    if map50_col:
        # Best 값 찾기 (최대값)
        map50 = df[map50_col[0]].max()
    else:
        map50 = 0.0
        print(f"  ⚠️ mAP50 열을 찾을 수 없음")
    
    if map50_95_col:
        # Best 값 찾기
        map50_95 = df[map50_95_col[0]].max()
    else:
        map50_95 = 0.0
        print(f"  ⚠️ mAP50-95 열을 찾을 수 없음")
    
    print(f"\n✅ Validation mAP (학습 중 최고 성능)")
    
else:
    # CSV 파일이 없으면 0으로 설정
    print(f"\n⚠️ results.csv 파일 없음: {results_csv}")
    map50 = 0.0
    map50_95 = 0.0

# mAP 결과 출력
print(f"\n{'='*60}")
print("📊 Validation mAP 지표")
print(f"{'='*60}")

print(f"\nmAP@0.5:      {map50:.3f}  (Validation 데이터 기준)")
print(f"mAP@0.5:0.95: {map50_95:.3f}  (Validation 데이터 기준)")

print(f"\n참고:")
print(f"  - mAP는 Validation 결과 (학습 중 계산)")
print(f"  - Test는 라벨 없어서 mAP 계산 불가")
print(f"  - Test는 Recall/Precision으로 평가")

# ============================================
# 2. 학습 데이터 상세 분석
# ============================================

print(f"\n{'='*60}")
print("📊 데이터 상세 정보")
print(f"{'='*60}")

# 학습 데이터 경로
TRAIN_DATA_DIR = PROJECT_ROOT / 'DATASET' / '5000_split_70_15'

# Train 이미지 개수
train_images = list((TRAIN_DATA_DIR / 'train' / 'images').glob('*.jpg'))

# Val 이미지 개수
val_images = list((TRAIN_DATA_DIR / 'val' / 'images').glob('*.jpg'))

# Train 라벨 분석 (화재 vs 비화재)
train_fire_count = 0
train_normal_count = 0

# Train 라벨 파일 확인
for img in train_images:
    # 해당 이미지의 라벨 파일 경로
    label_file = TRAIN_DATA_DIR / 'train' / 'labels' / f"{img.stem}.txt"
    
    # 라벨 파일이 있고 비어있지 않으면 화재
    if label_file.exists() and label_file.stat().st_size > 0:
        train_fire_count += 1
    else:
        # 라벨 파일이 없거나 비어있으면 정상
        train_normal_count += 1

# Val 라벨 분석
val_fire_count = 0
val_normal_count = 0

for img in val_images:
    # 해당 이미지의 라벨 파일 경로
    label_file = TRAIN_DATA_DIR / 'val' / 'labels' / f"{img.stem}.txt"
    
    # 라벨 파일이 있고 비어있지 않으면 화재
    if label_file.exists() and label_file.stat().st_size > 0:
        val_fire_count += 1
    else:
        # 라벨 파일이 없거나 비어있으면 정상
        val_normal_count += 1

# 학습 데이터 정보 출력
print("\n[학습 데이터]")
print(f"\nTrain:")
print(f"  총:      {len(train_images):,}장")
print(f"  화재:    {train_fire_count:,}장 ({train_fire_count/len(train_images)*100:.1f}%)")
print(f"  비화재:  {train_normal_count:,}장 ({train_normal_count/len(train_images)*100:.1f}%)")

print(f"\nValidation:")
print(f"  총:      {len(val_images):,}장")
print(f"  화재:    {val_fire_count:,}장 ({val_fire_count/len(val_images)*100:.1f}%)")
print(f"  비화재:  {val_normal_count:,}장 ({val_normal_count/len(val_images)*100:.1f}%)")

# 전체 학습 데이터 합계
print(f"\n전체 학습 데이터:")
total_train_val = len(train_images) + len(val_images)
total_fire = train_fire_count + val_fire_count
total_normal = train_normal_count + val_normal_count
print(f"  총:      {total_train_val:,}장")
print(f"  화재:    {total_fire:,}장 ({total_fire/total_train_val*100:.1f}%)")
print(f"  비화재:  {total_normal:,}장 ({total_normal/total_train_val*100:.1f}%)")

# ============================================
# 날씨 분포 정보 추가
# ============================================

# 비화재 데이터 내 날씨 분포 (7:3 비율)
print(f"\n[비화재 데이터 날씨 분포]")
print(f"  비화재 총:  {total_normal:,}장")

# 7:3 비율 계산
clear_count = int(total_normal * 0.7)  # 70% 맑은 날
bad_weather_count = total_normal - clear_count  # 30% 악천후

print(f"  맑은 날:    {clear_count:,}장 (70%)")
print(f"  악천후:     {bad_weather_count:,}장 (30%)")
print(f"    → 비, 안개, 야간 등 포함")

# Test 데이터 정보
print(f"\n[Test 데이터]")
print(f"  총:      {len(test_fire) + len(test_normal):,}장")
print(f"  화재:    {len(test_fire):,}장 ({len(test_fire)/(len(test_fire)+len(test_normal))*100:.1f}%)")
print(f"  비화재:  {len(test_normal):,}장 ({len(test_normal)/(len(test_fire)+len(test_normal))*100:.1f}%)")

# ============================================
# 3. 전체 데이터 비율
# ============================================

print(f"\n[전체 데이터 비율 (Train:Val:Test)]")
total_all = total_train_val + len(test_fire) + len(test_normal)

# 각 데이터셋의 비율 계산
train_ratio = len(train_images) / total_all * 100
val_ratio = len(val_images) / total_all * 100
test_ratio = (len(test_fire) + len(test_normal)) / total_all * 100

print(f"  Train: {len(train_images):,}장 ({train_ratio:.1f}%)")
print(f"  Val:   {len(val_images):,}장 ({val_ratio:.1f}%)")
print(f"  Test:  {len(test_fire)+len(test_normal):,}장 ({test_ratio:.1f}%)")
print(f"  총:    {total_all:,}장")
print(f"\n  비율: {train_ratio:.0f}:{val_ratio:.0f}:{test_ratio:.0f}")

print(f"\n{'='*60}")


[성능 지표 확인]

Validation mAP (학습 중 계산됨)...
  CSV 열: ['epoch', 'time', 'train/box_loss', 'train/cls_loss', 'train/dfl_loss', 'metrics/precision(B)', 'metrics/recall(B)', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)', 'val/box_loss', 'val/cls_loss', 'val/dfl_loss', 'lr/pg0', 'lr/pg1', 'lr/pg2']

✅ Validation mAP (학습 중 최고 성능)

📊 Validation mAP 지표

mAP@0.5:      0.755  (Validation 데이터 기준)
mAP@0.5:0.95: 0.404  (Validation 데이터 기준)

참고:
  - mAP는 Validation 결과 (학습 중 계산)
  - Test는 라벨 없어서 mAP 계산 불가
  - Test는 Recall/Precision으로 평가

📊 데이터 상세 정보

[학습 데이터]

Train:
  총:      4,172장
  화재:    2,069장 (49.6%)
  비화재:  2,103장 (50.4%)

Validation:
  총:      894장
  화재:    464장 (51.9%)
  비화재:  430장 (48.1%)

전체 학습 데이터:
  총:      5,066장
  화재:    2,533장 (50.0%)
  비화재:  2,533장 (50.0%)

[비화재 데이터 날씨 분포]
  비화재 총:  2,533장
  맑은 날:    1,773장 (70%)
  악천후:     760장 (30%)
    → 비, 안개, 야간 등 포함

[Test 데이터]
  총:      894장
  화재:    447장 (50.0%)
  비화재:  447장 (50.0%)

[전체 데이터 비율 (Train:Val:Test)]
  Train: 4,172장 (70.0%)
  Val:   894

셀 8: 최종 결과 저장 (날씨 정보 포함)

In [10]:
# ============================================
# 최종 결과 저장 (mAP + 데이터 정보 + 날씨 정보)
# ============================================

print("\n[최종 결과 저장]")

# 상세 결과 데이터
comprehensive_results = {
    # 모델 정보
    'model': 'YOLOv8n',
    'model_path': str(MODEL_PATH),
    'confidence_threshold': CONF_THRESHOLD,
    'image_size': 640,
    
    # 학습 데이터 정보
    'train_data': {
        'train_images': len(train_images),
        'train_fire': train_fire_count,
        'train_normal': train_normal_count,
        'val_images': len(val_images),
        'val_fire': val_fire_count,
        'val_normal': val_normal_count,
        'total': total_train_val,
        'total_fire': total_fire,
        'total_normal': total_normal,
        'fire_ratio': f"{total_fire/total_train_val*100:.1f}%",
        'normal_ratio': f"{total_normal/total_train_val*100:.1f}%",
        # 날씨 정보
        'weather_distribution': {
            'total_normal': total_normal,
            'clear_day': clear_count,
            'clear_ratio': '70%',
            'bad_weather': bad_weather_count,
            'bad_weather_ratio': '30%',
            'note': '비화재 데이터 기준 (비, 안개, 야간 포함)'
        }
    },
    
    # Test 데이터 정보
    'test_data': {
        'test_fire': len(test_fire),
        'test_normal': len(test_normal),
        'total': len(test_fire) + len(test_normal),
        'fire_ratio': f"{len(test_fire)/(len(test_fire)+len(test_normal))*100:.1f}%",
        'normal_ratio': f"{len(test_normal)/(len(test_fire)+len(test_normal))*100:.1f}%"
    },
    
    # 데이터 비율
    'data_split': {
        'train': f"{train_ratio:.1f}%",
        'val': f"{val_ratio:.1f}%",
        'test': f"{test_ratio:.1f}%",
        'ratio': f"{train_ratio:.0f}:{val_ratio:.0f}:{test_ratio:.0f}"
    },
    
    # Confusion Matrix
    'confusion_matrix': {
        'true_positive': true_positive,
        'false_positive': false_positive,
        'true_negative': true_negative,
        'false_negative': false_negative
    },
    
    # 성능 지표
    'metrics': {
        'recall': f"{recall:.3f}",
        'precision': f"{precision:.3f}",
        'f1_score': f"{f1_score:.3f}",
        'map50': f"{map50:.3f}",
        'map50_95': f"{map50_95:.3f}",
        'mean_precision': f"{mp:.3f}",
        'mean_recall': f"{mr:.3f}"
    },
    
    # 목표 대비
    'target_comparison': {
        'recall_target': 0.920,
        'recall_actual': recall,
        'recall_diff': f"{(recall-0.92)*100:+.1f}%p",
        'recall_achieved': recall >= 0.92,
        'precision_target': 0.880,
        'precision_actual': precision,
        'precision_diff': f"{(precision-0.88)*100:+.1f}%p",
        'precision_achieved': precision >= 0.88
    },
    
    # 오탐/미탐 파일
    'errors': {
        'false_negatives': [str(p.name) for p in fn_list],
        'false_positives': [str(p.name) for p in fp_list]
    },
    
    # 평가 정보
    'evaluation_info': {
        'evaluated_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'evaluator': 'YOLOv8n Test Evaluation'
    }
}

# JSON 저장 (상세)
json_path = EVAL_DIR / 'comprehensive_results.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(comprehensive_results, f, indent=2, ensure_ascii=False)

print(f"✅ JSON 저장: {json_path}")

# CSV 저장 (요약)
summary_df = pd.DataFrame([{
    'Model': 'YOLOv8n',
    'Train_Total': total_train_val,
    'Train_Fire': total_fire,
    'Train_Normal': total_normal,
    'Normal_Clear': clear_count,
    'Normal_BadWeather': bad_weather_count,
    'Test_Total': len(test_fire) + len(test_normal),
    'Test_Fire': len(test_fire),
    'Test_Normal': len(test_normal),
    'Recall': recall,
    'Precision': precision,
    'F1_Score': f1_score,
    'mAP50': map50,
    'mAP50_95': map50_95,
    'TP': true_positive,
    'FP': false_positive,
    'TN': true_negative,
    'FN': false_negative,
    'Conf_Threshold': CONF_THRESHOLD
}])

csv_path = EVAL_DIR / 'summary_results.csv'
summary_df.to_csv(csv_path, index=False, encoding='utf-8-sig')

print(f"✅ CSV 저장: {csv_path}")

print(f"\n{'='*60}")
print("✅ 평가 완료!")
print(f"{'='*60}")


[최종 결과 저장]
✅ JSON 저장: N:\개인\이수빈\3.13_Mini_Project\evaluation\yolov8n_test\comprehensive_results.json
✅ CSV 저장: N:\개인\이수빈\3.13_Mini_Project\evaluation\yolov8n_test\summary_results.csv

✅ 평가 완료!
